## Importing Datasets

In [1]:

import re
import numpy as np
import pandas as pd
import tensorflow as tf

from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Bidirectional,
    Dense, Dropout, Lambda, concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau


## Load Dataset

In [2]:


df = pd.read_csv('questions.csv')

df = df[['question1','question2','is_duplicate']].dropna()

print(df.shape)
df.head()


(404348, 3)


,question1,question2,is_duplicate
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


## Preprocessing (adapted from your original notebook)

In [3]:

contractions = {
    "can't": "can not",
    "won't": "will not",
    "i'm": "i am",
    "it's": "it is",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "you're": "you are",
    "they're": "they are",
    "we're": "we are",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "hasn't": "has not",
    "haven't": "have not",
    "hadn't": "had not",
    "wouldn't": "would not",
    "shouldn't": "should not",
    "couldn't": "could not",
    "that's": "that is",
    "what's": "what is",
    "there's": "there is",
    "who's": "who is",
    "let's": "let us"
}

def preprocess(text):
    text = str(text).lower()

    text = BeautifulSoup(text, "html.parser").get_text()


    for c, full in contractions.items():
        text = text.replace(c, full)


    text = text.replace("₹", " rupee ")
    text = text.replace("$", " dollar ")
    text = text.replace("%", " percentage ")
    text = text.replace("€", " euro ")


    text = re.sub(r'(\d+)000000000', r'\1b', text)
    text = re.sub(r'(\d+)000000', r'\1m', text)
    text = re.sub(r'(\d+)000', r'\1k', text)

    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Apply preprocessing
q1 = df['question1'].astype(str).apply(preprocess)
q2 = df['question2'].astype(str).apply(preprocess)
y = df['is_duplicate'].values

print(q1.iloc[0])
print(q2.iloc[0])


what is the step by step guide to invest in share market in india
what is the step by step guide to invest in share market


## Tokenization

In [4]:

VOCAB_SIZE = 50000
MAX_LEN = 35

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')

all_questions = list(q1) + list(q2)
tokenizer.fit_on_texts(all_questions)

q1_seq = tokenizer.texts_to_sequences(q1)
q2_seq = tokenizer.texts_to_sequences(q2)

q1_pad = pad_sequences(
    q1_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

q2_pad = pad_sequences(
    q2_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

print(q1_pad.shape)
print(q2_pad.shape)


(404348, 35)
(404348, 35)


## Train-Test Split

In [5]:

q1_train, q1_test, q2_train, q2_test, y_train, y_test = train_test_split(
    q1_pad,
    q2_pad,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(q1_train.shape, q1_test.shape)


(323478, 35) (80870, 35)


## Build Siamese BiLSTM

In [6]:

EMBED_DIM = 128

input1 = Input(shape=(MAX_LEN,))
input2 = Input(shape=(MAX_LEN,))

embedding_layer = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBED_DIM,
    mask_zero=True
)

x1 = embedding_layer(input1)
x2 = embedding_layer(input2)

shared_bilstm = Bidirectional(
    LSTM(128, dropout=0.2, recurrent_dropout=0.2)
)

x1 = shared_bilstm(x1)
x2 = shared_bilstm(x2)


diff = Lambda(
    lambda tensors: tf.abs(tensors[0] - tensors[1])
)([x1, x2])

merged = concatenate([x1, x2, diff])

dense = Dense(256, activation='relu')(merged)
dense = Dropout(0.3)(dense)

dense = Dense(128, activation='relu')(dense)
dense = Dropout(0.3)(dense)

output = Dense(1, activation='sigmoid')(dense)

model = Model(
    inputs=[input1, input2],
    outputs=output
)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 35)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 35)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 35, 128)   │  6,400,000 │ input_layer[0][0… │
│ (Embedding)         │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 35)        │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 35)        │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 256)       │    263,168 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0],  │
│                     │                   │            │ embedding[1][0],  │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 256)       │          0 │ bidirectional[0]… │
│                     │                   │            │ bidirectional[1]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 768)       │          0 │ bidirectional[0]… │
│ (Concatenate)       │                   │            │ bidirectional[1]… │
│                     │                   │            │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    196,864 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │        129 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,893,057 (26.29 MB)

 Trainable params: 6,893,057 (26.29 MB)

 Non-trainable params: 0 (0.00 B)

## Training

In [7]:

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        'best_siamese_bilstm.keras',
        monitor='val_loss',
        save_best_only=True
    ),
    ReduceLROnPlateau(
        monitor = 'val_loss',
        patience = 3,
        min_lr = 1e-6
    )
]

history = model.fit(
    [q1_train, q2_train],
    y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    callbacks=callbacks
)


Epoch 1/10
2275/2275 ━━━━━━━━━━━━━━━━━━━━ 1284s 557ms/step - accuracy: 0.7880 - loss: 0.4447 - val_accuracy: 0.7957 - val_loss: 0.4347 - learning_rate: 0.0010
Epoch 2/10
2275/2275 ━━━━━━━━━━━━━━━━━━━━ 1218s 535ms/step - accuracy: 0.8537 - loss: 0.3295 - val_accuracy: 0.8103 - val_loss: 0.4095 - learning_rate: 0.0010
Epoch 3/10
2275/2275 ━━━━━━━━━━━━━━━━━━━━ 1228s 540ms/step - accuracy: 0.8864 - loss: 0.2629 - val_accuracy: 0.8282 - val_loss: 0.3916 - learning_rate: 0.0010
Epoch 4/10
2275/2275 ━━━━━━━━━━━━━━━━━━━━ 1214s 533ms/step - accuracy: 0.9093 - loss: 0.2150 - val_accuracy: 0.8360 - val_loss: 0.4101 - learning_rate: 0.0010
Epoch 5/10
2275/2275 ━━━━━━━━━━━━━━━━━━━━ 1241s 542ms/step - accuracy: 0.9245 - loss: 0.1822 - val_accuracy: 0.8303 - val_loss: 0.4339 - learning_rate: 0.0010


## Evaluate

In [8]:

loss, accuracy = model.evaluate(
    [q1_test, q2_test],
    y_test,
    verbose=0
)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Loss: {loss:.4f}")


Test Accuracy: 0.8297
Test Loss: 0.3883


In [9]:

pred_prob = model.predict([q1_test, q2_test], verbose=0)
pred = (pred_prob > 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, pred_prob))
print()
print(classification_report(y_test, pred))

print("Confusion Matrix")
print(confusion_matrix(y_test, pred))


ROC-AUC: 0.9148550293652833

              precision    recall  f1-score   support

           0       0.91      0.81      0.86     51009
           1       0.73      0.86      0.79     29861

    accuracy                           0.83     80870
   macro avg       0.82      0.84      0.82     80870
weighted avg       0.84      0.83      0.83     80870

Confusion Matrix
[[41307  9702]
 [ 4068 25793]]


## Save tokenizer

In [10]:

import pickle

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Saved tokenizer.pkl")


Saved tokenizer.pkl
